# Práctica Día 05 — Fine-tune un ViT/ResNet preentrenado

Dataset sugerido: EuroSAT (imagen satélite, 10 clases). Para más casos: Food-101 (restauración), Stanford Cars, DocVQA (OCR).

Si Hugging Face Hub no es accesible desde tu red, descarga el modelo desde un mirror o usa torchvision como alternativa local.

In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
from datasets import load_dataset
ds = load_dataset('blanchon/EuroSAT_RGB', split='train').train_test_split(test_size=0.2, seed=0)
ds

In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification
proc = AutoImageProcessor.from_pretrained('google/vit-base-patch16-224')
labels = sorted(set(ds['train']['label']))
model = AutoModelForImageClassification.from_pretrained(
    'google/vit-base-patch16-224', num_labels=len(labels), ignore_mismatched_sizes=True)
sum(p.numel() for p in model.parameters() if p.requires_grad)  # parámetros entrenables

In [ ]:
from transformers import Trainer, TrainingArguments
def transform(batch):
    b = proc(batch['image'], return_tensors='pt'); b['labels'] = batch['label']; return b
ds = ds.with_transform(transform)
args = TrainingArguments(output_dir='out', num_train_epochs=2,
                          per_device_train_batch_size=16, learning_rate=2e-5,
                          weight_decay=0.01, lr_scheduler_type='cosine')
trainer = Trainer(model=model, args=args, train_dataset=ds['train'], eval_dataset=ds['test'])
trainer.train()

## Ejercicios

**5.1.** Cambia el backbone a `microsoft/resnet-50`. Compara accuracy y tiempo.
**5.2.** Congela los primeros 8 bloques del ViT (`for p in model.vit.encoder.layer[:8].parameters(): p.requires_grad = False`). Compara con el caso sin congelar.
**5.3 (sin IA).** Explica en 8 líneas: ¿por qué la tasa de aprendizaje correcta para fine-tuning ($2\cdot10^{-5}$) es mucho menor que para entrenar desde cero ($10^{-3}$)?

## Declaración de uso de IA

| Sección | Herramienta | Para qué |
|---|---|---|
| Código | … | … |
| Ejercicio 5.3 | Ninguna | — |